### Transform Payments Data
1. Extract Date and Time from payment_timestamp and create new columns payment_date and payment_time
2. Map payment_status to contain descriptive values
    (1-Success, 2-Pending, 3-Cancelled, 4-Failed)
3. Write transformed data to the Silver Schema

### 1. Extract Date and Time from payment_timestamp and create new columns payment_date and payment_time

In [0]:
from pyspark.sql.functions import *
pay_df=spark.sql('SELECT * FROM gizmobox.bronze.payments')

pay_time=pay_df.select(col('payment_id'),
                       col('order_id'),
                       col('payment_status'),
                       col('payment_method'),
                       date_format(col('payment_timestamp'),'yyyy-MM-dd').cast('date').alias('payment_date'),
                       date_format(col('payment_timestamp'),'HH:mm:ss').alias('payment_time'))

pay_time.display()
        

payment_id,order_id,payment_status,payment_method,payment_date,payment_time
35,13,2,PayPal,2024-12-17,05:25:34
36,16,4,Bank Transfer,2024-12-09,10:45:56
37,17,1,Bank Transfer,2024-12-02,13:50:20
38,24,1,PayPal,2024-12-22,08:30:55
39,27,2,Credit Card,2024-12-03,11:45:30
40,28,4,Credit Card,2024-12-27,05:40:10
41,32,1,PayPal,2024-12-29,09:20:40
42,37,2,Credit Card,2024-12-23,12:10:05
43,38,1,Credit Card,2024-12-26,20:15:15
44,44,1,Credit Card,2024-12-21,14:25:45


### 2. Map payment_status to contain descriptive values (1-Success, 2-Pending, 3-Cancelled, 4-Failed)

In [0]:
df_silver= pay_time.withColumn('payment_status', when(col('payment_status')==1,lit('Success')) \
    .when(col('payment_status')==2, lit('Pending')) \
    .when(col('payment_status')==3,lit('cancelled')) \
    .when(col('payment_status')==4,lit('failed')))


In [0]:
df_silver.display()

payment_id,order_id,payment_status,payment_method,payment_date,payment_time
35,13,Pending,PayPal,2024-12-17,05:25:34
36,16,failed,Bank Transfer,2024-12-09,10:45:56
37,17,Success,Bank Transfer,2024-12-02,13:50:20
38,24,Success,PayPal,2024-12-22,08:30:55
39,27,Pending,Credit Card,2024-12-03,11:45:30
40,28,failed,Credit Card,2024-12-27,05:40:10
41,32,Success,PayPal,2024-12-29,09:20:40
42,37,Pending,Credit Card,2024-12-23,12:10:05
43,38,Success,Credit Card,2024-12-26,20:15:15
44,44,Success,Credit Card,2024-12-21,14:25:45


### 3. Write transformed data to the Silver Schema

In [0]:
df_silver.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('gizmobox.silver.payments')

In [0]:
%sql
SELECT * FROM  gizmobox.silver.payments;

payment_id,order_id,payment_status,payment_method,payment_date,payment_time
2,10,Pending,Credit Card,2024-10-09,22:09:27
3,11,failed,Bank Transfer,2024-10-15,17:34:19
4,15,Success,Bank Transfer,2024-10-22,01:47:25
5,19,Pending,PayPal,2024-10-15,12:40:26
6,39,failed,PayPal,2024-10-31,21:39:19
7,55,Success,Bank Transfer,2024-10-27,05:49:16
8,59,Pending,PayPal,2024-10-25,02:51:05
9,72,cancelled,PayPal,2024-10-08,10:23:28
10,77,Success,Credit Card,2024-10-17,13:09:33
11,84,Success,PayPal,2024-10-27,03:37:42


In [0]:
%sql
DESCRIBE EXTENDED  gizmobox.silver.payments;

col_name,data_type,comment
payment_id,int,null
order_id,int,null
payment_status,string,null
payment_method,string,null
payment_date,date,null
payment_time,string,null
,,
# Delta Statistics Columns,,
Column Names,"payment_date, payment_method, payment_status, payment_id, payment_time, order_id",
Column Selection Method,first-32,
